In [6]:
import os
import json
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# グローバル定数
ZONES = ["TL","TC","TR","ML","MC","MR","BL","BC","BR"]
FEATURE_COLS = ["foot_enc", "match_time", "score_diff", "is_shootout", "home_away", "pressure_index"]

def ensure_dir(path: str) -> None:
    """ディレクトリが存在しない場合は作成します"""
    os.makedirs(path, exist_ok=True)

ensure_dir("models")
ensure_dir("data")
print("環境の準備が完了しました")

環境の準備が完了しました


In [7]:
def load_data(path: str = "hybrid_penalties.csv") -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"ファイルが見つかりません: {path}")
    
    df = pd.read_csv(path)
    need = set(["player_name", "zone_target"] + FEATURE_COLS)
    miss = need - set(df.columns)
    
    if miss:
        raise ValueError(f"不足しているカラムがあります: {sorted(miss)}")
    return df

# データの実行と確認
df = load_data("hybrid_penalties.csv")

print(f"データ件数: {len(df)}")
print("特徴量のデータ型:\n", df[FEATURE_COLS].dtypes)
df.head() # データの先頭5行を表示

データ件数: 2722
特徴量のデータ型:
 foot_enc            int64
match_time          int64
score_diff          int64
is_shootout         int64
home_away           int64
pressure_index    float64
dtype: object


,player_name,foot_enc,match_time,score_diff,is_shootout,home_away,pressure_index,zone_target,source
0,Victor Okoh Boniface,1,42,0,0,1,2.80,MR,real
1,Florian Wirtz,1,51,2,0,0,2.04,ML,real
2,Exequiel Alejandro Palacios,1,7,3,0,1,0.14,BR,real
3,Victor Okoh Boniface,1,15,-2,0,0,0.60,TC,real
4,Victor Okoh Boniface,1,62,0,0,1,4.13,BC,real


In [8]:
# 特徴量とラベルの準備
X = df[FEATURE_COLS].values
le = LabelEncoder()
y = le.fit_transform(df["zone_target"].values)

# データの分割 (層化抽出法を使用)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"学習データサイズ: {X_train.shape}")
print(f"テストデータサイズ: {X_test.shape}")
print(f"予測クラス一覧: {list(le.classes_)}")

学習データサイズ: (2177, 6)
テストデータサイズ: (545, 6)
予測クラス一覧: ['BC', 'BL', 'BR', 'MC', 'ML', 'MR', 'TC', 'TL', 'TR']


In [9]:
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(le.classes_),
    max_depth=5,
    learning_rate=0.08,
    n_estimators=300,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=4,
)

print("モデルの学習を開始します...")
model.fit(X_train, y_train, verbose=False)
print("学習が完了しました")

モデルの学習を開始します...
学習が完了しました


In [11]:
# 予測の実行
prob = model.predict_proba(X_test)
pred = prob.argmax(axis=1)

# 指標の計算
acc = accuracy_score(y_test, pred)
ll = log_loss(y_test, prob, labels=list(range(len(le.classes_))))

print("評価レポート")
print(f"- 正解率 (Accuracy): {acc:.4f}")
print(f"- マルチクラス対数損失 (LogLoss): {ll:.4f}")
print("\n--- 詳細な分類レポート ---")
print(classification_report(y_test, pred, target_names=le.classes_))


評価レポート
- 正解率 (Accuracy): 0.1394
- マルチクラス対数損失 (LogLoss): 2.4657

--- 詳細な分類レポート ---
              precision    recall  f1-score   support

          BC       0.08      0.05      0.06        44
          BL       0.18      0.27      0.21       117
          BR       0.21      0.26      0.23        96
          MC       0.00      0.00      0.00        37
          ML       0.16      0.13      0.14        69
          MR       0.04      0.03      0.04        59
          TC       0.06      0.05      0.05        39
          TL       0.03      0.02      0.03        42
          TR       0.12      0.07      0.09        42

    accuracy                           0.14       545
   macro avg       0.10      0.10      0.09       545
weighted avg       0.12      0.14      0.13       545



In [12]:
def compute_player_priors(df: pd.DataFrame, min_k: int = 5) -> dict:
    """各プレイヤーのシュート傾向（事前確率）を計算します"""
    counts = (
        df.groupby(["player_name", "zone_target"])
          .size()
          .unstack(fill_value=0)
          .reindex(columns=ZONES, fill_value=0)
    )
    totals = counts.sum(axis=1).astype(int)
    probs = (counts + 1.0).div((counts + 1.0).sum(axis=1), axis=0)

    priors = {}
    for player in probs.index:
        k = int(totals.loc[player])
        if k >= min_k:
            priors[player] = {"count": k, "probs": {z: float(probs.loc[player, z]) for z in ZONES}}
    return priors

# 実行と保存
priors = compute_player_priors(df, min_k=5)

joblib.dump(model, "models/zone_model.pkl")
joblib.dump(le, "models/label_encoder.pkl")
with open("models/player_priors.json", "w", encoding="utf-8") as f:
    json.dump(priors, f, ensure_ascii=False, indent=2)

print(f"モデルとプレイヤーデータ（計 {len(priors)} 名）を保存しました")

モデルとプレイヤーデータ（計 175 名）を保存しました
